In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms
from torchvision.utils import save_image
import os
import numpy as np
from PIL import Image

import os


In [2]:
os.environ['KAGGLE_USERNAME'] = 'ritik9929'
os.environ['KAGGLE_KEY'] = '<REDACTED: set your own key via Kaggle Secrets>'

In [3]:
class MRPNet(nn.Module):
    def __init__(self):
        super(MRPNet, self).__init__()
        self.encoder = nn.Sequential(
            nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(64, 64, kernel_size=3, stride=2, padding=1),
            nn.ReLU(inplace=True)
        )
        self.decoder = nn.Sequential(
            nn.ConvTranspose2d(64, 64, kernel_size=3, stride=2, padding=1, output_padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(64, 3, kernel_size=3, stride=1, padding=1)
        )
    def forward(self, x):
        x = self.encoder(x)
        x = self.decoder(x)
        return x

In [4]:
class Generator(nn.Module):
    def __init__(self):
        super(Generator, self).__init__()
        self.block1 = nn.Sequential(
            nn.Conv2d(3, 64, kernel_size=9, stride=1, padding=4),
            nn.ReLU(inplace=True)
        )
        self.block2 = nn.Sequential(
            nn.Conv2d(64, 64, kernel_size=3, stride=1, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True)
        )
        self.upsample = nn.Sequential(
            nn.Conv2d(64, 256, kernel_size=3, stride=1, padding=1),
            nn.PixelShuffle(2),
            nn.ReLU(inplace=True),
            nn.Conv2d(64, 256, kernel_size=3, stride=1, padding=1),
            nn.PixelShuffle(2),
            nn.ReLU(inplace=True)
        )
        self.block3 = nn.Conv2d(64, 3, kernel_size=9, stride=1, padding=4)
    def forward(self, x):
        x = self.block1(x)
        residual = x
        x = self.block2(x)
        x = x + residual
        x = self.upsample(x)
        x = self.block3(x)
        return x

In [5]:
class CombinedModel(nn.Module):
    def __init__(self):
        super(CombinedModel, self).__init__()
        self.denoiser = MRPNet()
        self.super_resolver = Generator()
    def forward(self, x):
        x = self.denoiser(x)
        x = self.super_resolver(x)
        return x

In [6]:
class LowLightDataset(Dataset):
    def __init__(self, clean_dir, noisy_dir, transform=None):
        self.noisy_dir = noisy_dir
        self.transform = transform
        self.clean_dir = clean_dir
        self.image_list = os.listdir(noisy_dir)
    def __len__(self):
        return len(self.image_list)
    def __getitem__(self, idx):
        image_name = self.image_list[idx]
        name, img_no = image_name.split("_")
        
        noise_path = os.path.join(self.noisy_dir, self.image_list[idx])
        clean_path = os.path.join(self.clean_dir, "gt_"+img_no)
        
        noise_img = Image.open(noise_path).convert("RGB")
        clean_image = Image.open(clean_path).convert("RGB")
        
        if self.transform:
            clean_image = self.transform(clean_image)
            noise_img = self.transform(noise_img)
        return clean_image, noise_img

In [7]:
class TestLowLightDataset(Dataset):
    def __init__(self, image_dir, transform=None):
        self.image_dir = image_dir
        self.transform = transform
        self.image_list = os.listdir(image_dir)
    def __len__(self):
        return len(self.image_list)
    def __getitem__(self, idx):
        img_path = os.path.join(self.image_dir, self.image_list[idx])
        image = Image.open(img_path).convert("RGB")

        if self.transform:
            image = self.transform(image)

        return image, self.image_list[idx]

In [8]:
transform = transforms.Compose([
    transforms.ToTensor(),
])

In [9]:
train_clean = r'/kaggle/input/dlp-jan-2025-nppe-3/archive/train/gt'
train_noisy = r'/kaggle/input/dlp-jan-2025-nppe-3/archive/train/train'
val_clean = r'/kaggle/input/dlp-jan-2025-nppe-3/archive/val/gt'
val_noisy = r'/kaggle/input/dlp-jan-2025-nppe-3/archive/val/val'
test = r'/kaggle/input/dlp-jan-2025-nppe-3/archive/test'

In [10]:
train_dataset = LowLightDataset(train_clean,train_noisy, transform=transform)
val_dataset = LowLightDataset(val_clean,val_noisy, transform=transform)
test_dataset = TestLowLightDataset(test, transform = transform)

In [11]:
train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True, num_workers=4)
val_loader = DataLoader(val_dataset, batch_size=8, shuffle=False, num_workers=4)
test_loader = DataLoader(test_dataset, batch_size =8, shuffle = False, num_workers = 4)

In [12]:
def train_model(model, dataloader, optimizer, criterion, scaler, device):
    model.train()
    running_loss = 0.0
    for clean_images, noisy_images in dataloader:
        clean_images,noisy_images = clean_images.to(device), noisy_images.to(device)
        optimizer.zero_grad()
        with torch.cuda.amp.autocast():
            outputs = model(noisy_images)
            loss = criterion(outputs, clean_images)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        running_loss += loss.item()

    return running_loss / len(dataloader)

In [13]:
def validate_model(model, dataloader, criterion, device):
  model.eval()
  running_loss = 0.0
  with torch.no_grad():
    for clean_images, noisy_images in dataloader:
        clean_images = clean_images.to(device)
        noisy_images = noisy_images.to(device)
        outputs = model(noisy_images)
        loss = criterion(outputs, clean_images)
        running_loss += loss.item()
  return running_loss / len(dataloader)

In [14]:
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
model = CombinedModel()

model = nn.DataParallel(model)  
model = model.to(device)

criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=1e-4)
scaler = torch.cuda.amp.GradScaler()

<ipython-input-14-58318179a8f2>:9: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler()


In [15]:
model.load_state_dict(torch.load('/kaggle/input/best_model/pytorch/default/1/best_model.pth'))

<ipython-input-15-0452ade7c90a>:1: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load('/kaggle/input/best_model/pytorch/default/1/best_model.pth'

<All keys matched successfully>

In [16]:
num_epochs = 70
best_val_loss = float("inf")

for epoch in range(num_epochs):

  train_loss = train_model(model, train_loader, optimizer, criterion, scaler, device)
  val_loss = validate_model(model, val_loader, criterion, device)

  if val_loss < best_val_loss:
      best_val_loss = val_loss
      torch.save(model.state_dict(), "best_model.pth")
  print(f"Epoch {epoch+1}/{num_epochs}, Train Loss: {train_loss:.9f}, Val Loss: {val_loss:.9f}")

<ipython-input-12-03aeaeae6202>:7: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():


Epoch 1/70, Train Loss: 0.000257115, Val Loss: 0.000146550
Epoch 2/70, Train Loss: 0.000236259, Val Loss: 0.000144600
Epoch 3/70, Train Loss: 0.000237143, Val Loss: 0.000152770
Epoch 4/70, Train Loss: 0.000235493, Val Loss: 0.000146836
Epoch 5/70, Train Loss: 0.000235827, Val Loss: 0.000165063
Epoch 6/70, Train Loss: 0.000237955, Val Loss: 0.000156258
Epoch 7/70, Train Loss: 0.000236188, Val Loss: 0.000143969
Epoch 8/70, Train Loss: 0.000239773, Val Loss: 0.000159415
Epoch 9/70, Train Loss: 0.000236657, Val Loss: 0.000145601
Epoch 10/70, Train Loss: 0.000237608, Val Loss: 0.000172982
Epoch 11/70, Train Loss: 0.000237241, Val Loss: 0.000176271
Epoch 12/70, Train Loss: 0.000235978, Val Loss: 0.000154076
Epoch 13/70, Train Loss: 0.000235736, Val Loss: 0.000155604
Epoch 14/70, Train Loss: 0.000235658, Val Loss: 0.000151336
Epoch 15/70, Train Loss: 0.000235575, Val Loss: 0.000177845
Epoch 16/70, Train Loss: 0.000237787, Val Loss: 0.000147582
Epoch 17/70, Train Loss: 0.000235093, Val Loss: 0

In [17]:
os.makedirs('/kaggle/working/test_final',exist_ok=True)
final_path = "/kaggle/working/test_final"

In [18]:
def test_prediction(model, test_loader, device, path):
    model.eval()
    with torch.no_grad():
        with tqdm(total = len(test_loader), desc = 'Testing', unit = 'batch') as tepoch:
            for noise_imgs, img_name in test_loader:
                noise_imgs = noise_imgs.to(device)

                outputs = model(noise_imgs)

                for idx in range(outputs.shape[0]):
                    predicted_img = outputs[idx].squeeze(0).cpu()
                    
                    out_path = os.path.join(path, f"{img_name[idx]}")
                    save_image(predicted_img, out_path)

                tepoch.update(1)

In [19]:
from tqdm import tqdm

In [20]:
model.load_state_dict(torch.load("/kaggle/working/best_model.pth"))

<ipython-input-20-55445dfdbf8b>:1: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load("/kaggle/working/best_model.pth"))


<All keys matched successfully>

In [21]:
test_prediction(model,test_loader,device, final_path)

Testing: 100%|██████████| 8/8 [00:16<00:00,  2.04s/batch]


In [22]:
import os
import numpy as np
import pandas as pd
from PIL import Image

def images_to_csv(folder_path, output_csv):
    data_rows = []
    for filename in os.listdir(folder_path):
        if filename.endswith(('.png', '.jpg', '.jpeg', '.bmp', '.tiff')):
            image_path = os.path.join(folder_path, filename)
            image = Image.open(image_path).convert('L') 
            image_array = np.array(image).flatten()[::8]
            image_id = filename.split('.')[0].replace('test_', 'gt_')
            data_rows.append([image_id, *image_array])
    column_names = ['ID'] + [f'pixel_{i}' for i in range(len(data_rows[0]) - 1)]
    df = pd.DataFrame(data_rows, columns=column_names)
    df.to_csv(output_csv, index=False)
    print(f'Successfully saved to {output_csv}')

folder_path = final_path
output_csv = 'submission.csv'
images_to_csv(folder_path, output_csv)

from kaggle.api.kaggle_api_extended import KaggleApi

api = KaggleApi()
api.authenticate()

api.competition_submit(
    file_name='submission.csv',  # Make sure this file exists in your current working directory
    message="Auto submission Done",
    competition="dlp-jan-2025-nppe-3"
)

print("Submission complete!")

Successfully saved to submission.csv


100%|██████████| 15.3M/15.3M [00:00<00:00, 26.6MB/s]


Submission complete!
